# Homework 5: Model Deployment

## Objective

In this assignment, I use the F1 dataset to build two predictive models, log them in MLflow, and store each model's predictions in my own database tables. The target variable is `points`, and I use race-level and pit-stop-related features to build the modeling dataset.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import shutil
import tempfile

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Part 1. Load and Clean the Data

In this section, I load the relevant F1 datasets and clean the basic data types. Since the raw files contain missing values represented as `\N`, I convert them into null values during loading.

In [0]:
base_path = "/Volumes/gr5069/raw/f1_data"

df_drivers = (
    spark.read
    .option("header", True)
    .option("nullValue", "\\N")
    .csv(f"{base_path}/drivers.csv")
    .withColumn("driverId", F.col("driverId").cast("int"))
    .withColumn("dob", F.to_date("dob", "yyyy-MM-dd"))
)

df_races = (
    spark.read
    .option("header", True)
    .option("nullValue", "\\N")
    .csv(f"{base_path}/races.csv")
    .withColumn("raceId", F.col("raceId").cast("int"))
    .withColumn("year", F.col("year").cast("int"))
    .withColumn("round", F.col("round").cast("int"))
    .withColumn("date", F.to_date("date", "yyyy-MM-dd"))
)

df_results = (
    spark.read
    .option("header", True)
    .option("nullValue", "\\N")
    .csv(f"{base_path}/results.csv")
    .withColumn("resultId", F.col("resultId").cast("int"))
    .withColumn("raceId", F.col("raceId").cast("int"))
    .withColumn("driverId", F.col("driverId").cast("int"))
    .withColumn("constructorId", F.col("constructorId").cast("int"))
    .withColumn("number", F.col("number").cast("int"))
    .withColumn("grid", F.col("grid").cast("int"))
    .withColumn("position", F.col("position").cast("int"))
    .withColumn("positionOrder", F.col("positionOrder").cast("int"))
    .withColumn("points", F.col("points").cast("double"))
    .withColumn("laps", F.col("laps").cast("int"))
    .withColumn("milliseconds", F.col("milliseconds").cast("bigint"))
    .withColumnRenamed("milliseconds", "race_milliseconds")
)

df_pitstops = (
    spark.read
    .option("header", True)
    .option("nullValue", "\\N")
    .csv(f"{base_path}/pit_stops.csv")
    .withColumn("raceId", F.col("raceId").cast("int"))
    .withColumn("driverId", F.col("driverId").cast("int"))
    .withColumn("stop", F.col("stop").cast("int"))
    .withColumn("lap", F.col("lap").cast("int"))
    .withColumn("milliseconds", F.col("milliseconds").cast("bigint"))
    .withColumnRenamed("milliseconds", "pit_milliseconds")
)

## Part 2. Create the Modeling Dataset

I aggregate pit stop features and join them with the race results table and selected race information. This creates one row per driver-race observation.

In [0]:
df_pit_features = (
    df_pitstops
    .groupBy("raceId", "driverId")
    .agg(
        F.count("*").alias("pit_stop_count"),
        F.sum("pit_milliseconds").alias("total_pit_time"),
        F.avg("pit_milliseconds").alias("avg_pit_time")
    )
)

df_model = (
    df_results
    .join(
        df_races.select("raceId", "year", "round"),
        on="raceId",
        how="left"
    )
    .join(
        df_pit_features,
        on=["raceId", "driverId"],
        how="left"
    )
    .fillna({
        "pit_stop_count": 0,
        "total_pit_time": 0,
        "avg_pit_time": 0
    })
)

model_cols = [
    "raceId",
    "driverId",
    "grid",
    "laps",
    "year",
    "round",
    "pit_stop_count",
    "total_pit_time",
    "avg_pit_time",
    "points"
]

df_model_final = (
    df_model
    .select(*model_cols)
    .dropna()
)

display(df_model_final.limit(10))
print("Final row count:", df_model_final.count())

## Part 3. Prepare Train/Test Data

I convert the final Spark DataFrame to pandas so I can use scikit-learn models and log them with MLflow.

In [0]:
pdf = df_model_final.toPandas()

feature_cols = [
    "grid",
    "laps",
    "year",
    "round",
    "pit_stop_count",
    "total_pit_time",
    "avg_pit_time"
]

target_col = "points"

X = pdf[feature_cols]
y = pdf[target_col]

id_cols = pdf[["raceId", "driverId"]]

X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, id_cols, test_size=0.2, random_state=42
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## Part 4. Set Up MLflow Experiment

I create an MLflow experiment for Homework 5. Both models will be logged under this experiment.

In [0]:
experiment_name = "/Users/wx2337@columbia.edu/f1_homework5_mlflow"
mlflow.set_experiment(experiment_name)

print("Using MLflow experiment:", experiment_name)

## Part 5. Create Two Prediction Tables in My Own Database

The assignment requires storing predictions from each model in two separate tables. I create both tables in my own database/schema.

In [0]:
display(spark.sql("SHOW SCHEMAS"))

In [0]:
# CHANGE THIS TO YOUR OWN DATABASE / SCHEMA NAME
my_database = "default"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {my_database}")

table_model1 = f"{my_database}.hw5_model1_predictions"
table_model2 = f"{my_database}.hw5_model2_predictions"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {table_model1} (
    raceId INT,
    driverId INT,
    actual_points DOUBLE,
    predicted_points DOUBLE,
    residual DOUBLE,
    model_name STRING,
    run_id STRING
)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {table_model2} (
    raceId INT,
    driverId INT,
    actual_points DOUBLE,
    predicted_points DOUBLE,
    residual DOUBLE,
    model_name STRING,
    run_id STRING
)
""")

print("Created tables:")
print(table_model1)
print(table_model2)

## Part 6. Train and Log Model 1: Random Forest Regressor

The first model is a Random Forest Regressor. I log its hyperparameters, model object, metrics, and artifacts in MLflow.

In [0]:
rf_params = {
    "n_estimators": 200,
    "max_depth": 10,
    "min_samples_split": 5,
    "random_state": 42
}

with mlflow.start_run(run_name="hw5_random_forest") as rf_run:
    rf_model = RandomForestRegressor(
        n_estimators=rf_params["n_estimators"],
        max_depth=rf_params["max_depth"],
        min_samples_split=rf_params["min_samples_split"],
        random_state=rf_params["random_state"],
        n_jobs=-1
    )
    
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_test)
    
    rf_mae = mean_absolute_error(y_test, rf_pred)
    rf_mse = mean_squared_error(y_test, rf_pred)
    rf_rmse = np.sqrt(rf_mse)
    rf_r2 = r2_score(y_test, rf_pred)
    
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("target", target_col)
    mlflow.log_param("features", ", ".join(feature_cols))
    mlflow.log_param("n_estimators", rf_params["n_estimators"])
    mlflow.log_param("max_depth", rf_params["max_depth"])
    mlflow.log_param("min_samples_split", rf_params["min_samples_split"])
    mlflow.log_param("random_state", rf_params["random_state"])
    
    mlflow.log_metric("mae", float(rf_mae))
    mlflow.log_metric("mse", float(rf_mse))
    mlflow.log_metric("rmse", float(rf_rmse))
    mlflow.log_metric("r2", float(rf_r2))
    
    mlflow.sklearn.log_model(rf_model, "random_forest_model")
    
    temp_dir = tempfile.mkdtemp()
    
    try:
        rf_pred_df = pd.DataFrame({
            "raceId": id_test["raceId"].values,
            "driverId": id_test["driverId"].values,
            "actual_points": y_test.values,
            "predicted_points": rf_pred,
            "residual": y_test.values - rf_pred
        })
        
        pred_path = os.path.join(temp_dir, "rf_predictions.csv")
        rf_pred_df.to_csv(pred_path, index=False)
        mlflow.log_artifact(pred_path)
        
        fi_df = pd.DataFrame({
            "feature": feature_cols,
            "importance": rf_model.feature_importances_
        }).sort_values("importance", ascending=False)
        
        fi_path = os.path.join(temp_dir, "rf_feature_importance.csv")
        fi_df.to_csv(fi_path, index=False)
        mlflow.log_artifact(fi_path)
        
        plt.figure(figsize=(8, 5))
        plt.scatter(rf_pred, y_test.values - rf_pred, alpha=0.5)
        plt.axhline(y=0)
        plt.xlabel("Predicted Points")
        plt.ylabel("Residuals")
        plt.title("Random Forest Residual Plot")
        plot_path = os.path.join(temp_dir, "rf_residual_plot.png")
        plt.savefig(plot_path, bbox_inches="tight")
        plt.close()
        mlflow.log_artifact(plot_path)
        
    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)
    
    rf_run_id = rf_run.info.run_id

print("Random Forest run ID:", rf_run_id)
print("RF metrics:", rf_mae, rf_mse, rf_rmse, rf_r2)

## Part 7. Write Model 1 Predictions to the Database

I store the Random Forest predictions in the first prediction table in my own database.

In [0]:
rf_predictions_to_save = pd.DataFrame({
    "raceId": id_test["raceId"].values,
    "driverId": id_test["driverId"].values,
    "actual_points": y_test.values,
    "predicted_points": rf_pred,
    "residual": y_test.values - rf_pred,
    "model_name": "RandomForestRegressor",
    "run_id": rf_run_id
})

spark_rf_predictions = spark.createDataFrame(rf_predictions_to_save)
spark_rf_predictions.write.mode("overwrite").saveAsTable(table_model1)

print("Saved Random Forest predictions to:", table_model1)
display(spark.table(table_model1).limit(10))

## Part 8. Train and Log Model 2: Decision Tree Regressor

The second model is a Decision Tree Regressor. I log its hyperparameters, model object, metrics, and artifacts in MLflow.

In [0]:
dt_params = {
    "max_depth": 8,
    "min_samples_split": 10,
    "random_state": 42
}

with mlflow.start_run(run_name="hw5_decision_tree") as dt_run:
    dt_model = DecisionTreeRegressor(
        max_depth=dt_params["max_depth"],
        min_samples_split=dt_params["min_samples_split"],
        random_state=dt_params["random_state"]
    )
    
    dt_model.fit(X_train, y_train)
    dt_pred = dt_model.predict(X_test)
    
    dt_mae = mean_absolute_error(y_test, dt_pred)
    dt_mse = mean_squared_error(y_test, dt_pred)
    dt_rmse = np.sqrt(dt_mse)
    dt_r2 = r2_score(y_test, dt_pred)
    
    mlflow.log_param("model_type", "DecisionTreeRegressor")
    mlflow.log_param("target", target_col)
    mlflow.log_param("features", ", ".join(feature_cols))
    mlflow.log_param("max_depth", dt_params["max_depth"])
    mlflow.log_param("min_samples_split", dt_params["min_samples_split"])
    mlflow.log_param("random_state", dt_params["random_state"])
    
    mlflow.log_metric("mae", float(dt_mae))
    mlflow.log_metric("mse", float(dt_mse))
    mlflow.log_metric("rmse", float(dt_rmse))
    mlflow.log_metric("r2", float(dt_r2))
    
    mlflow.sklearn.log_model(dt_model, "decision_tree_model")
    
    temp_dir = tempfile.mkdtemp()
    
    try:
        dt_pred_df = pd.DataFrame({
            "raceId": id_test["raceId"].values,
            "driverId": id_test["driverId"].values,
            "actual_points": y_test.values,
            "predicted_points": dt_pred,
            "residual": y_test.values - dt_pred
        })
        
        pred_path = os.path.join(temp_dir, "dt_predictions.csv")
        dt_pred_df.to_csv(pred_path, index=False)
        mlflow.log_artifact(pred_path)
        
        fi_df = pd.DataFrame({
            "feature": feature_cols,
            "importance": dt_model.feature_importances_
        }).sort_values("importance", ascending=False)
        
        fi_path = os.path.join(temp_dir, "dt_feature_importance.csv")
        fi_df.to_csv(fi_path, index=False)
        mlflow.log_artifact(fi_path)
        
        plt.figure(figsize=(8, 5))
        plt.scatter(dt_pred, y_test.values - dt_pred, alpha=0.5)
        plt.axhline(y=0)
        plt.xlabel("Predicted Points")
        plt.ylabel("Residuals")
        plt.title("Decision Tree Residual Plot")
        plot_path = os.path.join(temp_dir, "dt_residual_plot.png")
        plt.savefig(plot_path, bbox_inches="tight")
        plt.close()
        mlflow.log_artifact(plot_path)
        
    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)
    
    dt_run_id = dt_run.info.run_id

print("Decision Tree run ID:", dt_run_id)
print("DT metrics:", dt_mae, dt_mse, dt_rmse, dt_r2)

## Part 9. Write Model 2 Predictions to the Database

I store the Decision Tree predictions in the second prediction table in my own database.

In [0]:
dt_predictions_to_save = pd.DataFrame({
    "raceId": id_test["raceId"].values,
    "driverId": id_test["driverId"].values,
    "actual_points": y_test.values,
    "predicted_points": dt_pred,
    "residual": y_test.values - dt_pred,
    "model_name": "DecisionTreeRegressor",
    "run_id": dt_run_id
})

spark_dt_predictions = spark.createDataFrame(dt_predictions_to_save)
spark_dt_predictions.write.mode("overwrite").saveAsTable(table_model2)

print("Saved Decision Tree predictions to:", table_model2)
display(spark.table(table_model2).limit(10))

## Part 10. Compare the Two Models

I compare the evaluation metrics from both models to understand which model performed better on the test set.

In [0]:
comparison_df = pd.DataFrame([
    {
        "model": "RandomForestRegressor",
        "mae": rf_mae,
        "mse": rf_mse,
        "rmse": rf_rmse,
        "r2": rf_r2
    },
    {
        "model": "DecisionTreeRegressor",
        "mae": dt_mae,
        "mse": dt_mse,
        "rmse": dt_rmse,
        "r2": dt_r2
    }
])

display(comparison_df)

## Part 11. Conclusion

In this homework, I built two predictive models using the F1 dataset and logged both models in MLflow. I used `points` as the target variable and created a modeling dataset from race results, race information, and aggregated pit stop features.

The two models were a `RandomForestRegressor` and a `DecisionTreeRegressor`. Based on the evaluation results, the Random Forest model performed better overall. It achieved a lower MAE, MSE, and RMSE, as well as a higher R² value than the Decision Tree model. Specifically, the Random Forest model achieved an RMSE of about 2.55 and an R² of about 0.665, while the Decision Tree model achieved an RMSE of about 2.64 and an R² of about 0.639.

In addition, I created two prediction tables and stored the predictions from each model in the database. This workflow demonstrates a complete pipeline including data preparation, model training, experiment tracking with MLflow, and storing prediction outputs in database tables.